# 🌍 Exploratory Analysis — World Bank Education Statistics

This notebook presents an initial exploratory analysis of the **World Bank Education Statistics (EdStats)** dataset used in the Global Education AI Agent project.

The goal is to understand the structure of the raw dataset before running the complete Python pipeline.


## 1. Import Libraries

The analysis uses `pandas` for data exploration and `matplotlib` for simple visualizations.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


## 2. Define Dataset Path

The raw EdStats files must be located in the `data/raw/` directory.


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
EDSTATS_DATA_FILE = RAW_DATA_DIR / "EdStatsData.csv"

EDSTATS_DATA_FILE


## 3. Load Raw Dataset

The main file `EdStatsData.csv` contains the historical values of the education indicators.


In [ ]:
df = pd.read_csv(EDSTATS_DATA_FILE)
df.head()


## 4. Dataset Shape

The shape shows the number of rows and columns in the dataset.


In [ ]:
df.shape


## 5. Main Columns

The dataset includes country information, indicator information and one column for each year.


In [ ]:
df.columns.tolist()[:15]


## 6. Countries and Indicators

This section checks how many countries and indicators are available in the dataset.


In [ ]:
summary = {
    "countries": df["Country Code"].nunique(),
    "indicators": df["Indicator Code"].nunique(),
    "rows": df.shape[0],
    "columns": df.shape[1],
}

summary


## 7. Missing Values

The EdStats dataset contains many missing values because not all countries report all indicators for all years.


In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values.head(10)


## 8. Example Indicator Selection

This section filters a sample indicator used by the project: public expenditure on education as a percentage of GDP.


In [ ]:
indicator_code = "SE.XPD.TOTL.GD.ZS"
countries = ["BRA", "CHL", "ARG", "USA", "FIN"]

sample_df = df[
    (df["Indicator Code"] == indicator_code)
    & (df["Country Code"].isin(countries))
].copy()

sample_df[["Country Name", "Country Code", "Indicator Name", "Indicator Code"]].head()


## 9. Transform Sample to Long Format

The project pipeline transforms the original wide format into a long analytical format.


In [ ]:
year_columns = [col for col in sample_df.columns if str(col).isdigit()]

long_sample = sample_df.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=year_columns,
    var_name="year",
    value_name="value",
)

long_sample["year"] = long_sample["year"].astype(int)
long_sample["value"] = pd.to_numeric(long_sample["value"], errors="coerce")
long_sample = long_sample.dropna(subset=["value"])
long_sample = long_sample[(long_sample["year"] >= 2000) & (long_sample["year"] <= 2020)]

long_sample.head()


## 10. Simple Visualization

The chart below shows the historical evolution of the selected indicator for the selected countries.


In [ ]:
plt.figure(figsize=(12, 6))

for country_code, group in long_sample.groupby("Country Code"):
    plt.plot(group["year"], group["value"], marker="o", label=country_code)

plt.title("Government Expenditure on Education (% of GDP)")
plt.xlabel("Year")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Conclusion

This exploratory notebook confirms that the EdStats dataset contains a large number of countries, indicators and historical values.

The complete analysis is executed through the modular pipeline available in the `src/` directory, especially through:

```bash
python -m src.main --skip-openai
```

The notebook is intended only as an initial exploration and does not replace the production pipeline.
